# RQ1: Engagement Differences Between Conditions

Onboarding sample: participants who completed the onboarding survey and were active on the feed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pingouin as pg

FIG_DIR = "../tables_figures/"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "mathtext.fontset": "custom",
    "mathtext.rm": "Arial",
    "mathtext.it": "Arial:italic",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

ARM_LABELS = {"K": "Control", "M": "Treatment"}


def describe_by_arm(df, value_col):
    """n, mean, SD, skewness and excess kurtosis of value_col per condition."""
    return (
        df.groupby("arm")[value_col]
        .agg(
            n="count",
            mean="mean",
            sd="std",
            skew="skew",
            kurtosis=lambda x: x.kurt(),
        )
        .rename(index=ARM_LABELS)
        .round(2)
    )


activity = pd.read_csv("../data/onboarding_sample_round1.csv", sep=";", decimal=",", encoding="utf-8-sig")
print(activity.shape)

In [ ]:
# Keep only the days a participant could have been active: from their first day (where their
# rows start) up to and including the day they received the offboarding survey
activity = activity[activity["day"] <= activity["offboarding_received_date"]]
print(len(activity), "rows |", activity["handle"].nunique(), "participants")

# Descriptives for Activity

## Active Days

In [ ]:
# One row per participant: number of days on which they were active
active_days = (
    activity
    .groupby(["handle", "arm"], as_index=False)["active"]
    .sum()
    .rename(columns={"active": "n_active_days"})
)
active_days.shape

In [ ]:
describe_by_arm(active_days, "n_active_days")

In [ ]:
# All participants together
active_days["n_active_days"].agg(["mean", "std", "min", "max"]).round(2)

In [ ]:
min_days = active_days["n_active_days"].min()
max_days = active_days["n_active_days"].max()
bins = np.arange(min_days - 0.5, max_days + 1.5, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (arm, label) in zip(axes, ARM_LABELS.items()):
    subset = active_days.loc[active_days["arm"] == arm, "n_active_days"]
    ax.hist(subset, bins=bins,  edgecolor="black")
    ax.set_title(label)
    ax.set_xlabel("Number of Active Days")
    stats_text = (
        f"$n$ = {len(subset)}\n"
        f"$M$ = {subset.mean():.2f}\n"
        f"$SD$ = {subset.std():.2f}\n"
        f"Skewness = {subset.skew():.2f}\n"
        f"Kurtosis = {subset.kurt():.2f}"
    )
    ax.text(0.97, 0.97, stats_text, transform=ax.transAxes,
            ha="right", va="top", fontsize=10)
axes[0].set_ylabel("Number of Participants")
fig.tight_layout()
fig.savefig(FIG_DIR + "active_days_by_condition.png")
plt.show()

## Engagement

In [ ]:
# One row per participant: total reactions over the study period
engagement = (
    activity
    .groupby(["handle", "arm"], as_index=False)["reactions"]
    .sum()
    .rename(columns={"reactions": "n_reactions"})
)
engagement.shape

In [ ]:
max_reactions = engagement["n_reactions"].max()
bins = np.arange(-0.5, max_reactions + 10, 10)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (arm, label) in zip(axes, ARM_LABELS.items()):
    subset = engagement.loc[engagement["arm"] == arm, "n_reactions"]
    ax.hist(subset, bins=bins, edgecolor="black")
    ax.set_title(label)
    ax.set_xlabel("Number of Reactions")
    stats_text = (
        f"$n$ = {len(subset)}\n"
        f"$M$ = {subset.mean():.2f}\n"
        f"$SD$ = {subset.std():.2f}\n"
        f"Skewness = {subset.skew():.2f}\n"
        f"Kurtosis = {subset.kurt():.2f}"
    )
    ax.text(0.97, 0.97, stats_text, transform=ax.transAxes,
            ha="right", va="top", fontsize=10)
axes[0].set_ylabel("Number of Participants")
fig.tight_layout()
fig.savefig(FIG_DIR + "engagement_by_condition.png")
plt.show()

In [ ]:
describe_by_arm(engagement, "n_reactions")

In [ ]:
# All participants together
engagement["n_reactions"].agg(["mean", "std", "min", "max"]).round(2)

# T-Tests for condition differences

In [ ]:
display(pg.normality(active_days, dv="n_active_days", group="arm"))
display(pg.homoscedasticity(active_days, dv="n_active_days", group="arm"))

In [ ]:
control = active_days.loc[active_days["arm"] == "K", "n_active_days"]
treatment = active_days.loc[active_days["arm"] == "M", "n_active_days"]

display(pg.ttest(treatment, control, correction=True))

In [ ]:
display(pg.normality(engagement, dv="n_reactions", group="arm"))
display(pg.homoscedasticity(engagement, dv="n_reactions", group="arm"))

In [ ]:
control = engagement.loc[engagement["arm"] == "K", "n_reactions"]
treatment = engagement.loc[engagement["arm"] == "M", "n_reactions"]

display(pg.ttest(treatment, control, correction=True))